In [1]:
import torch
import numpy as np

In [2]:
a=torch.FloatTensor(3,2)
a

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])

In [3]:
torch.zeros(3,4)

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

In [4]:
a.zero_()

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])

In [5]:
torch.FloatTensor([[1,2,3],[4,5,6]])

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [8]:
n= np.zeros(shape=(3,2),dtype=np.float32)
print(n)
b=torch.tensor(n)
print(b)

[[0. 0.]
 [0. 0.]
 [0. 0.]]
tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])


In [10]:
a=torch.tensor([1,2,3])
print(a)
s=a.sum()
print(s)
s.item()
torch.tensor(1)

tensor([1, 2, 3])
tensor(6)


tensor(1)

In [11]:
a=torch.FloatTensor([2,3])
print(a)
ca= a.to('mps')
print(ca)

tensor([2., 3.])
tensor([2., 3.], device='mps:0')


In [12]:
ca.device

device(type='mps', index=0)

In [14]:
v1= torch.tensor([1.0,1.0],requires_grad=True)
v2= torch.tensor([2.0,2.0])
v_sum=v1+v2
print(v_sum)
v_res=(v_sum*2).sum()
v_res

tensor([3., 3.], grad_fn=<AddBackward0>)


tensor(12., grad_fn=<SumBackward0>)

In [17]:
print(v1.is_leaf,v2.is_leaf)
print(v_sum.is_leaf)
print(v1.requires_grad,v2.requires_grad)
print(v_sum.requires_grad)
print(v_res.requires_grad)


True True
False
True False
True
True


In [18]:
v_res.backward()
v1.grad

tensor([2., 2.])

In [19]:
v2.grad

In [20]:
p=torch.nn.Linear(2,5)
v=torch.FloatTensor([1,2])
p(v)

tensor([-0.3516, -1.2694,  1.6350, -0.4629,  0.0256], grad_fn=<ViewBackward0>)

In [22]:
import torch.nn as nn
s=nn.Sequential(
    nn.Linear(2,5),
    nn.ReLU(),
    nn.Linear(5,20),
    nn.ReLU(),
    nn.Linear(20,10),
    nn.Dropout(p=0.5),
    nn.Softmax(dim=1)
)
s

Sequential(
  (0): Linear(in_features=2, out_features=5, bias=True)
  (1): ReLU()
  (2): Linear(in_features=5, out_features=20, bias=True)
  (3): ReLU()
  (4): Linear(in_features=20, out_features=10, bias=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Softmax(dim=1)
)

In [24]:
s(torch.FloatTensor([[1,2]]))

tensor([[0.0804, 0.1768, 0.0850, 0.0849, 0.0931, 0.1388, 0.0931, 0.0539, 0.0922,
         0.1016]], grad_fn=<SoftmaxBackward0>)

In [26]:
class OurModule(nn.Module):
    def __init__(self,num_inputs,num_classes,dropout_prob=0.3):
        super(OurModule,self).__init__()
        self.pipe=nn.Sequential(
            nn.Linear(num_inputs,5),
            nn.ReLU(),
            nn.Linear(5,20),
            nn.ReLU(),
            nn.Linear(20,num_classes),
            nn.Dropout(dropout_prob),
            nn.Softmax(dim=1)
        )
    def forward(self,x):
        return self.pipe(x)
if __name__=="__main__":
    net= OurModule(num_inputs=2,num_classes=3)
    print(net)
    v=torch.FloatTensor([[2,3]])
    out=net(v)
    print(out)
    print("MPS availability is %s"%torch.mps.is_available())
    if torch.mps.is_available():
        print("Data from MPS: %s" % out.to('mps'))

OurModule(
  (pipe): Sequential(
    (0): Linear(in_features=2, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
    (5): Dropout(p=0.3, inplace=False)
    (6): Softmax(dim=1)
  )
)
tensor([[0.2897, 0.2599, 0.4504]], grad_fn=<SoftmaxBackward0>)
MPS availability is True
Data from MPS: tensor([[0.2897, 0.2599, 0.4504]], device='mps:0', grad_fn=<ToCopyBackward0>)


In [ ]:
"""
Common blueprint for training loop:
1 for batch_x, batch_y in iterate_batches(data, batch_size=N):
2 batch_x_t = torch.tensor(batch_x)
3 batch_y_t = torch.tensor(batch_y)
4 out_t = net(batch_x_t)
5 loss_t = loss_function(out_t, batch_y_t).
6 loss_t.backward()
7 optimizer.step()
8 optimizer.zero_grad()

"""

In [30]:
import math
from torch.utils.tensorboard.writer import SummaryWriter
if __name__=="__main__":
    writer=SummaryWriter()
    funcs={"sin":math.sin,"cos":math.cos,"tan":math.tan}
    for angle in range(-360,360):
        angle_rad=angle*math.pi/180
        for name, fun in funcs.items():
            val=fun(angle_rad)
            writer.add_scalar(name,val,angle)
    writer.close()